## Question

Can we prepare a balanced dataset that teaches a model to route music-service messages to the correct part of the application?

Data preparation is the foundation of fine-tuning. A model can only learn the categories, boundaries, and mistakes represented in its training examples. Duplicate leakage, inconsistent labels, weak category definitions, or unrepresentative messages can make evaluation misleading and produce an unreliable classifier. For that reason, data preparation has its own notebook, where the dataset can be inspected and validated before any training begins.

The classifier needs to distinguish six intents: recommendation requests, recommendation-explanation requests, customer support, partnerships, junk, and messages that need human review. An explanation request can be sent to the RAG workflow built in the previous notebook, while the other categories follow their own routes.

## What This Notebook Covers

This notebook focuses only on preparing trustworthy data. It shows how to label messages, remove exact duplicates, create balanced splits, and save the records in the format needed for local training and evaluation.

It does not train, convert, or deploy a model. Those steps introduce new ideas and are easier to understand after the data pipeline is working.

## Hypothesis

If we define one source file per intent, remove exact duplicates before splitting, and split every intent separately, the resulting training, validation, and test sets should contain all six categories without exact-message leakage between splits.

Exporting complete conversations as MLX-LM chat records should produce training data with the expected answer, while evaluation records should keep the answer outside the messages sent to Ollama.

## Environment Setup

Create or activate the project environment with Python 3.11 or newer and install the shared dependencies:

```sh
python3.11 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

On an Apple Silicon Mac, install the separate MLX-LM training dependencies:

```sh
pip install -r requirements-mlx.txt
```

Install Ollama separately and pull the corresponding quantized build used for the prompting baseline:

```sh
ollama pull qwen2.5:0.5b-instruct
```

MLX-LM will teach `Qwen/Qwen2.5-0.5B-Instruct` from the prepared examples. Ollama will run the original model for a baseline and later run the fine-tuned model. Ollama does not perform the training itself. This preparation notebook creates the files needed by both tools without loading the model weights.

## Experiment

We will load twelve example messages for each intent, normalize them for duplicate detection, and assign numeric labels from the order of `source_files`. Each intent is shuffled with a fixed random seed and split approximately 80%/10%/10%.

After saving the labelled Hugging Face dataset, we will export complete training and validation conversations as JSONL for MLX-LM. Evaluation conversations omit the assistant answer and store the expected category separately for comparison with Ollama predictions.

### Build the labelled dataset

The first code cell performs the main data-quality work. It accepts execution from either the repository root or the notebook directory, checks every CSV for a `text` column and non-empty messages, and rejects identical messages assigned to conflicting labels.

`casefold()` and whitespace normalization make duplicate comparison insensitive to capitalization and repeated spaces. The original message text is preserved in the saved dataset. `ClassLabel` stores the category names with the numeric labels so later notebooks do not have to reproduce the list order. `utf-8-sig` accepts ordinary UTF-8 CSVs as well as files containing a byte-order mark.

In [190]:
import csv
import json
import random
from pathlib import Path

from datasets import ClassLabel, Dataset, DatasetDict, Features, Value, load_from_disk

notebook_dir = Path.cwd()
if not (notebook_dir / "intentData").is_dir():
    notebook_dir = notebook_dir / "embeddings-and-search"

data_dir = notebook_dir / "data"
label_names = [
    "Recommendation_Request",
    "Explanation_Request",
    "Customer_Support",
    "Partnership",
    "Junk",
    "Human_Review",
]
source_files = [
    "recommendation_request.csv",
    "explanation_request.csv",
    "customer_support.csv",
    "partnership.csv",
    "junk.csv",
    "human_review.csv",
]

rng = random.Random(42)
split_records = {"train": [], "validation": [], "test": []}
seen = {}

for label, filename in enumerate(source_files):
    records = []
    with (notebook_dir / "intentData" / filename).open(
        newline="", encoding="utf-8-sig"
    ) as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames or "text" not in reader.fieldnames:
            raise ValueError(f"{filename} must contain a text column")
        # DictReader already consumes the header.
        for row in reader:
            text = row["text"].strip()
            if not text:
                raise ValueError(f"Empty message in {filename}")
            key = " ".join(text.casefold().split())
            if key in seen:
                if seen[key] != label:
                    raise ValueError("The same message has conflicting category labels")
                continue
            seen[key] = label
            # File order defines numeric labels consistently across all splits.
            records.append({"text": text, "label": label})

    if len(records) < 10:
        raise ValueError(f"{filename} needs at least 10 unique messages")
    rng.shuffle(records)
    train_end = int(len(records) * 0.8)
    validation_end = train_end + int(len(records) * 0.1)
    split_records["train"].extend(records[:train_end])
    split_records["validation"].extend(records[train_end:validation_end])
    split_records["test"].extend(records[validation_end:])

for records in split_records.values():
    rng.shuffle(records)

dataset_features = Features({
    "text": Value("string"),
    "label": ClassLabel(names=label_names),
})
dataset = DatasetDict({
    name: Dataset.from_list(records, features=dataset_features)
    for name, records in split_records.items()
})
data_dir.mkdir(parents=True, exist_ok=True)
dataset.save_to_disk(str(data_dir / "music_service_intents_labeled"))
print(dataset)

Saving the dataset (1/1 shards): 100%|██████████| 12/12 [00:00<00:00, 5774.63 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 54
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 6
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
})


### Reload the labelled data

Reloading the saved artifact checks that it can be read independently and recovers the category names from its `ClassLabel` schema. The system instruction defines the six valid outputs and sends unclear or equally plausible messages to `Human_Review`.

In [191]:
dataset = load_from_disk(str(data_dir / "music_service_intents_labeled"))
label_names = dataset["train"].features["label"].names

In [192]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 54
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 6
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12
    })
})

In [193]:
message_system = """
You route messages sent to a music service.
Classify each message as Recommendation_Request, Explanation_Request, Customer_Support, Partnership, Junk, or Human_Review.
Use Human_Review when the intent is unclear or more than one route is equally plausible.
Return only the category name, without an explanation.
""".strip()

### Build the MLX-LM training and validation records

MLX-LM accepts chat data as one JSON object per line with a `messages` field. Each record contains a complete conversation: the system instruction, the user's message, and the correct category as the assistant response.

The records remain structured messages. MLX-LM applies Qwen's chat template when it tokenizes them for training, so this notebook does not pre-render template text.

In [194]:
def convert_to_training_messages(dataset):
    return [
        {
            "messages": [
                {"role": "system", "content": message_system},
                {
                    "role": "user",
                    "content": f"Classify: {record['text']}",
                },
                {
                    "role": "assistant",
                    "content": label_names[record["label"]],
                },
            ]
        }
        for record in dataset
    ]

In [195]:
train_records = convert_to_training_messages(dataset["train"])
validation_records = convert_to_training_messages(dataset["validation"])

### Build the Ollama evaluation records

Evaluation records contain only the system and user messages sent to Ollama. The expected category is stored in a separate `label` field, outside the conversation, so the model cannot see the answer.

These records are for classification accuracy and error analysis. They are not supplied to MLX-LM during training or validation.

In [196]:
def convert_to_evaluation_messages(dataset):
    return [
        {
            "messages": [
                {"role": "system", "content": message_system},
                {
                    "role": "user",
                    "content": f"Classify: {record['text']}",
                },
            ],
            "label": label_names[record["label"]],
        }
        for record in dataset
    ]

In [197]:
evaluation_records = convert_to_evaluation_messages(dataset["test"])

### Export the local training files

Write one JSON object per line to `data/music_service_intents_mlx`. MLX-LM recognizes `train.jsonl` and `valid.jsonl` automatically. `evaluation.jsonl` is kept alongside them for the Ollama comparison but is not a training input.

In [198]:
mlx_data_dir = data_dir / "music_service_intents_mlx"
mlx_data_dir.mkdir(parents=True, exist_ok=True)


def save_jsonl(path, records):
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


jsonl_files = {
    "train": mlx_data_dir / "train.jsonl",
    "valid": mlx_data_dir / "valid.jsonl",
    "evaluation": mlx_data_dir / "evaluation.jsonl",
}
save_jsonl(jsonl_files["train"], train_records)
save_jsonl(jsonl_files["valid"], validation_records)
save_jsonl(jsonl_files["evaluation"], evaluation_records)

## Results

Reload the JSONL files to confirm that every line is valid and the saved counts match the prepared splits. With the supplied twelve messages per category, the dataset contains 72 unique messages:

| File | Per intent | Total | Purpose |
| --- | ---: | ---: | --- |
| `train.jsonl` | 9 | 54 | MLX-LM fine-tuning conversations |
| `valid.jsonl` | 1 | 6 | Validation loss during training |
| `evaluation.jsonl` | 2 | 12 | Unseen Ollama classification evaluation |

Because the split is performed within each source file, every file contains all six intents in equal proportions.

In [199]:
def load_jsonl(path):
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


saved_records = {
    name: load_jsonl(path)
    for name, path in jsonl_files.items()
}
for name, records in saved_records.items():
    print(f"{name}: {len(records)} records")

{
    "training_example": saved_records["train"][0],
    "evaluation_example": saved_records["evaluation"][0],
}

train: 54 records
valid: 6 records
evaluation: 12 records


{'training_example': {'messages': [{'role': 'system',
    'content': 'You route messages sent to a music service.\nClassify each message as Recommendation_Request, Explanation_Request, Customer_Support, Partnership, Junk, or Human_Review.\nUse Human_Review when the intent is unclear or more than one route is equally plausible.\nReturn only the category name, without an explanation.'},
   {'role': 'user',
    'content': 'Classify: Which themes does this song share with my favourites?'},
   {'role': 'assistant', 'content': 'Explanation_Request'}]},
 'evaluation_example': {'messages': [{'role': 'system',
    'content': 'You route messages sent to a music service.\nClassify each message as Recommendation_Request, Explanation_Request, Customer_Support, Partnership, Junk, or Human_Review.\nUse Human_Review when the intent is unclear or more than one route is equally plausible.\nReturn only the category name, without an explanation.'},
   {'role': 'user',
    'content': 'Classify: Can you rec

## Analysis

The preparation meets the notebook's structural goal: each intent is represented in every split, exact normalized duplicates cannot cross split boundaries, and the fixed seed makes the result reproducible. `ClassLabel` preserves the category mapping in the labelled artifact, while the evaluation JSONL stores readable category names outside the model messages.

These checks establish dataset organization, not classifier quality. A complete evaluation still needs an Ollama prompting baseline, MLX-LM fine-tuning, predictions on the unchanged evaluation records, and per-category error analysis.

## Failure Cases

Exact duplicate detection does not catch paraphrases such as “Why was this suggested?” and “What caused this recommendation?”. Closely related paraphrases in different splits could make evaluation look better than real-world performance.

Some intents naturally overlap. A message about a recommended song failing to play could be an explanation request or customer support request. `Human_Review` gives the system a safe route for ambiguity, but its boundary depends on consistent labelling.

The examples are small and written for this lesson. Twelve messages per intent are enough to demonstrate the pipeline but not enough to establish dependable production behavior. Real messages may contain spelling errors, multiple requests, adversarial junk, languages not represented here, or references that require conversation history.

## Conclusion

The notebook produces two reusable artifacts. `music_service_intents_labeled` preserves the cleaned messages, balanced splits, and named labels. `music_service_intents_mlx` contains MLX-LM chat files for training and validation plus separate evaluation conversations for Ollama.

This preparation extends the song recommendation application with a routing dataset. It does not replace the recommender or the RAG explanation workflow; it prepares a classifier that can decide which workflow should handle a message.

## What Comes After Data Preparation

The prepared files make it possible to ask a practical question: **does local fine-tuning classify these messages more reliably than prompting the original model?** The remaining learning path is:

1. **Measure the starting point.** Ask the original Qwen model in Ollama to classify the messages in `evaluation.jsonl` and save its answers. This is the baseline.
2. **Fine-tune locally.** Use MLX-LM to train Qwen with `train.jsonl` while checking progress with `valid.jsonl`. Prompt masking will make training focus on the assistant's category answer rather than the repeated instructions.
3. **Evaluate fairly.** Run the fine-tuned model on the same evaluation messages and compare its answers with the hidden labels. Overall accuracy and the mistakes in each category will show whether training helped.
4. **Learn deployment later.** Model adapters, fusion, GGUF conversion, quantization, and Ollama import are separate deployment concepts. They can be introduced when the trained model is ready to run.

If fine-tuning produces little improvement, the extra work may not be justified. If it improves the important routing errors, the classifier can become the application entry point: `Recommendation_Request` goes to the recommender, `Explanation_Request` goes to the existing RAG workflow, and the remaining labels go to their corresponding queues.